# T2.5 - DBRepo Load and 3NF Verification

**Owner:** C
**Dataset:** Road Traffic Accidents (STATS19) 2009-2013, North Yorkshire
**Source file:** `data/raw/1-rta2009to2013fortableau.csv` (8358 rows, 42 columns)

1. **3NF verification** - proves with evidence from the CSV that the T2.1 schema
correctly models the data in Third Normal Form, and documents any data quality
issues found.

2. **Data loading** - transforms the CSV into the 5 target tables and loads them
into DBRepo via the REST API in FK-safe order.

3. **View verification** - confirms all five T2.4 views return non-empty,
correct results.

**Note on lookup tables:** The following tables were already seeded in T2.1 and
are not loaded here: `severity_type`, `road_surface_condition`, `light_condition`,
`weather_condition`, `special_condition_at_site`, `carriageway_hazard`, `road_type`,
`road_class`, `junction_detail`, `junction_control`, `pedestrian_crossing_control`,
`pedestrian_crossing_facility`, `reporting_authority`, `police_force`.
This notebook loads only the remaining 5 tables:
`local_authority_district`, `lower_super_output_area`, `output_area`,
`accident`, and `accident_road`.

In [20]:
import sys
!{sys.executable} -m pip install requests pandas dbrepo

In [21]:
import json
import time
import requests
import pandas as pd
from pathlib import Path
from requests.auth import HTTPBasicAuth
from dbrepo.RestClient import RestClient

ENDPOINT    = "https://test.dbrepo.tuwien.ac.at"
USERNAME    = "sravanthi"
PASSWORD    = "Srikanth@123"
DATABASE_ID = "f36ef3e2-1aee-4526-b3ea-82f661a9261a"

CSV_PATH = Path("../data/raw/1-rta2009to2013fortableau.csv")

AUTH    = HTTPBasicAuth(USERNAME, PASSWORD)
HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
client  = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)

print("Authenticated as:", client.whoami())

IDS      = json.loads(Path("dbrepo_ids.json").read_text())
TABLE_IDS = IDS["tables"]
VIEW_IDS  = IDS.get("views", {})
print(f"Loaded {len(TABLE_IDS)} table IDs, {len(VIEW_IDS)} view IDs")

sravanthi
Authenticated as: sravanthi
Loaded 19 table IDs, 5 view IDs


In [22]:
# Load raw CSV
df = pd.read_csv(CSV_PATH)
print(f"Raw CSV: {df.shape[0]} rows × {df.shape[1]} columns")

Raw CSV: 8358 rows × 42 columns


In [23]:
# 3NF Check 1: redundant LAD columns
assert (df['LAD11CD'] == df['LAD12CD']).all(), "LAD11CD != LAD12CD"
assert (df['LAD11NM'] == df['LAD12NM']).all(), "LAD11NM != LAD12NM"
assert df['LAD12NMW'].isna().all(), "LAD12NMW is not empty"
print("Check 1 PASS — LAD11* columns are fully redundant with LAD12*; LAD12NMW is empty.")
print("  Schema correctly consolidates to lad12_code, eliminating the transitive dependency.")

Check 1 PASS — LAD11* columns are fully redundant with LAD12*; LAD12NMW is empty.
  Schema correctly consolidates to lad12_code, eliminating the transitive dependency.


In [24]:
# 3NF Check 2: OA → LSOA functional dependency
lsoa_per_oa = df.groupby('OA11CD')['LSOA11CD'].nunique()
ambiguous   = lsoa_per_oa[lsoa_per_oa > 1]
print(f"Check 2 — OA→LSOA functional dependency")
print(f"  Unambiguous OAs : {(lsoa_per_oa == 1).sum()}")
print(f"  Ambiguous OAs   : {len(ambiguous)}  (ONS boundary revision artefact)")
if not ambiguous.empty:
    detail = df[df['OA11CD'].isin(ambiguous.index)][['OA11CD','LSOA11CD','LSOA11NM']].drop_duplicates().sort_values('OA11CD')
    print(detail.to_string(index=False))
    print("  Resolution: assign the most-frequent LSOA per OA (majority vote).")

Check 2 — OA→LSOA functional dependency
  Unambiguous OAs : 1466
  Ambiguous OAs   : 6  (ONS boundary revision artefact)
   OA11CD  LSOA11CD         LSOA11NM
E00140833        18   Harrogate 015A
E00140833        28   Harrogate 020A
E00140993        18   Harrogate 016B
E00140993        14   Harrogate 016A
E00141105        20   Harrogate 004C
E00141105        22   Harrogate 004B
E00141143        12   Harrogate 002A
E00141143        22   Harrogate 002B
E00141923        28 Scarborough 005E
E00141923        26 Scarborough 004F
E00142234        18       Selby 004C
E00142234        16       Selby 004D
  Resolution: assign the most-frequent LSOA per OA (majority vote).


In [25]:
# 3NF Check 3: referential integrity of all coded columns
LOOKUP_VALID = {
    'Severity'  : {1,2,3,4},
    'Road_cond' : {0,1,2,3,4,5,9},
    'Visibility': {1,4,5,6,7,10,11,12,13},
    'Weather'   : {0,1,2,3,4,5,6,7,8,9},
    'Spcond'    : {0,1,2,3,4,5,6,7,9},
    'Carr_haz'  : {0,1,2,3,6,7,9},
    'Road_type' : {0,1,2,3,6,7,9},
    'Junct_det' : {0,1,2,3,5,6,7,8,9,99},
    'Junct_ctrl': {0,1,2,3,4,9},
    'Cross_ctrl': {0,1,2,9},
    'Cross_fac' : {0,1,4,5,7,8,9},
    'Reportedat': {1,2,3},
    'Roadclass1': {1,2,3,4,5,6,9},
}
all_ok = True
for col, valid in LOOKUP_VALID.items():
    orphans = set(df[col].dropna().unique()) - valid
    status  = "PASS" if not orphans else f"ORPHAN VALUES: {orphans}"
    print(f"  {col:<15s}  {status}")
    if orphans:
        all_ok = False
r2_orphans = set(df['Roadclass2'].dropna().astype(int).unique()) - {1,2,3,4,5,6,9}
print(f"  {'Roadclass2':<15s}  {'PASS' if not r2_orphans else r2_orphans}")
print()
print("Check 3", "PASS" if all_ok else "FAIL", "— all coded values resolve to lookup table rows.")

  Severity         PASS
  Road_cond        PASS
  Visibility       PASS
  Weather          PASS
  Spcond           PASS
  Carr_haz         PASS
  Road_type        PASS
  Junct_det        PASS
  Junct_ctrl       PASS
  Cross_ctrl       PASS
  Cross_fac        PASS
  Reportedat       PASS
  Roadclass1       PASS
  Roadclass2       PASS

Check 3 PASS — all coded values resolve to lookup table rows.


In [26]:
# 3NF Check 4: repeating group resolved in accident_road
road2_present = df['Roadclass2'].notna().sum()
print(f"Check 4 — repeating group (Road1 / Road2)")
print(f"  Accidents with Road 1 only    : {len(df) - road2_present}")
print(f"  Accidents with Road 1 + Road 2: {road2_present}")
print(f"  Total accident_road rows      : {len(df) + road2_present}")
print("  PASS — repeating group eliminated by accident_road child table.")

Check 4 — repeating group (Road1 / Road2)
  Accidents with Road 1 only    : 5002
  Accidents with Road 1 + Road 2: 3356
  Total accident_road rows      : 11714
  PASS — repeating group eliminated by accident_road child table.


In [27]:
#Data preparation
oa_primary_lsoa = (
    df.groupby('OA11CD')['LSOA11CD']
      .agg(lambda x: x.value_counts().idxmax())
      .reset_index()
      .rename(columns={'LSOA11CD': 'lsoa_id_primary'})
)
print(f"Primary OA→LSOA mapping: {len(oa_primary_lsoa)} OAs")

Primary OA→LSOA mapping: 1472 OAs


In [28]:
lad_df = (
    df[['LAD12CD','LAD12CDO','LAD12NM']].drop_duplicates()
    .rename(columns={'LAD12CD':'lad12_code','LAD12CDO':'lad12_code_old','LAD12NM':'lad_name'})
    .reset_index(drop=True)
)
assert len(lad_df) == 7
print(f"local_authority_district: {len(lad_df)} rows")

local_authority_district: 7 rows


In [29]:
lsoa_df = (
    df[['LSOA11CD','LSOA11NM','LAD12CD']].drop_duplicates()
    .sort_values('LAD12CD')
    .groupby('LSOA11CD', as_index=False).first()
    .rename(columns={'LSOA11CD':'lsoa_id','LSOA11NM':'lsoa_name','LAD12CD':'lad12_code'})
    .reset_index(drop=True)
)
assert len(lsoa_df) == 39
print(f"lower_super_output_area: {len(lsoa_df)} rows")

lower_super_output_area: 39 rows


In [30]:
oa_df = (
    df[['OA11CD','AREA_HECT','RU_DEF_COD','RU_DEF_DES','Rural or Urban']].drop_duplicates('OA11CD')
    .merge(oa_primary_lsoa, on='OA11CD', how='left')
    .rename(columns={
        'OA11CD':'oa11_code', 'lsoa_id_primary':'lsoa_id',
        'AREA_HECT':'area_hectares', 'RU_DEF_COD':'rurality_code',
        'RU_DEF_DES':'rurality_description', 'Rural or Urban':'rural_urban'
    })[['oa11_code','lsoa_id','area_hectares','rurality_code','rurality_description','rural_urban']]
    .reset_index(drop=True)
)
assert len(oa_df) == 1472
assert oa_df['rural_urban'].isin({'Rural','Urban'}).all()
print(f"output_area: {len(oa_df)} rows")

output_area: 1472 rows


In [31]:
acc_df = df[[
    'Police_ref','Date','Time','Day','Easting','Northing','Longitude','Latitude',
    'Severity','Road_cond','Visibility','Weather','Spcond','Carr_haz',
    'Casualties','Vehicles','Road_type','Speed_lim','Junct_det','Junct_ctrl',
    'Cross_ctrl','Cross_fac','Pol_force','OA11CD','Reportedat'
]].copy()
acc_df.columns = [
    'police_ref','accident_date','accident_time','day_of_week',
    'easting','northing','longitude','latitude',
    'severity_id','road_cond_id','light_condition_id','weather_condition_id',
    'special_condition_id','carriageway_hazard_id',
    'casualties','vehicles','road_type_id','speed_limit_mph',
    'junction_detail_id','junction_control_id',
    'crossing_control_id','crossing_facility_id',
    'force_id','oa11_code','reporting_authority_id'
]
acc_df['accident_date'] = pd.to_datetime(acc_df['accident_date'], dayfirst=True).dt.strftime('%Y-%m-%d')
assert len(acc_df) == 8358
assert acc_df['police_ref'].nunique() == len(acc_df)
print(f"accident: {len(acc_df)} rows")

accident: 8358 rows


In [32]:
road1 = df[['Police_ref','Roadclass1','Roadnum1']].copy()
road1['road_sequence'] = 1
road1 = road1.rename(columns={'Police_ref':'police_ref','Roadclass1':'road_class_id','Roadnum1':'road_number'})

road2 = df[df['Roadclass2'].notna()][['Police_ref','Roadclass2','Roadnum2']].copy()
road2['road_sequence'] = 2
road2['Roadclass2']   = road2['Roadclass2'].astype(int)
road2['Roadnum2']     = road2['Roadnum2'].fillna(0).astype(int)
road2 = road2.rename(columns={'Police_ref':'police_ref','Roadclass2':'road_class_id','Roadnum2':'road_number'})

road_df = pd.concat([road1, road2], ignore_index=True)[['police_ref','road_sequence','road_class_id','road_number']]
assert len(road_df) == 11714
print(f"accident_road: {len(road_df)} rows  (road1={len(road1)}, road2={len(road2)})")

accident_road: 11714 rows  (road1=8358, road2=3356)


In [33]:
# Load into db repo
def load_table(name, dataframe, settle=15):
    """Load a DataFrame into DBRepo via import_table_data. Waits settle seconds for async processing."""
    table_id = TABLE_IDS[name]
    print(f"  Loading {name:<35s} ({len(dataframe)} rows) ... ", end="", flush=True)
    try:
        client.import_table_data(
            database_id = DATABASE_ID,
            table_id    = table_id,
            dataframe   = dataframe,
        )
        time.sleep(settle)
        print("OK")
    except Exception as exc:
        print(f"FAILED: {exc}")
        raise

In [34]:
print("Loading geographic hierarchy...")
load_table("local_authority_district", lad_df,  settle=5)
load_table("lower_super_output_area",  lsoa_df, settle=8)
load_table("output_area",              oa_df,   settle=15)

Loading geographic hierarchy...
  Loading local_authority_district            (7 rows) ... OK
  Loading lower_super_output_area             (39 rows) ... OK
  Loading output_area                         (1472 rows) ... OK


In [39]:
print("Loading accident fact table (8358 rows)...")
load_table("accident", acc_df, settle=45)

Loading accident fact table (8358 rows)...
  Loading accident                            (8358 rows) ... OK


In [37]:
print("Loading accident_road (11714 rows)...")
load_table("accident_road", road_df, settle=30)

Loading accident_road (11714 rows)...
  Loading accident_road                       (11714 rows) ... OK


In [41]:
def get_row_count(table_id):
    r = requests.get(
        f"{ENDPOINT}/api/v1/database/{DATABASE_ID}/table/{table_id}/data",
        params={"page": 0, "size": 1},
        auth=AUTH, headers=HEADERS, timeout=30
    )
    if not r.ok:
        return f"HTTP {r.status_code}"
    body = r.json()
    if isinstance(body, dict):
        return body.get("pagination", {}).get("total", body.get("total", "?"))
    elif isinstance(body, list):
        r2 = requests.get(
            f"{ENDPOINT}/api/v1/database/{DATABASE_ID}/table/{table_id}/data",
            params={"page": 0, "size": 100000},
            auth=AUTH, headers=HEADERS, timeout=60
        )
        body2 = r2.json()
        return len(body2) if isinstance(body2, list) else body2.get("pagination", {}).get("total", "?")
    return "?"

EXPECTED = {
    "local_authority_district": 7,
    "lower_super_output_area":  39,
    "output_area":              1472,
    "accident":                 8358,
    "accident_road":            11714,
}

print(f"  {'Table':<35s}  {'Expected':>9s}  {'Actual':>9s}  Status")
print("-" * 65)
all_pass = True
for tname, expected in EXPECTED.items():
    actual = get_row_count(TABLE_IDS[tname])
    ok     = actual == expected
    if not ok:
        all_pass = False
    print(f"  {tname:<35s}  {expected:>9d}  {str(actual):>9s}  {'PASS' if ok else 'MISMATCH'}")

print()
print("All row counts match." if all_pass else "Some counts differ — check for duplicate loads.")


  Table                                 Expected     Actual  Status
-----------------------------------------------------------------
  local_authority_district                     7          7  PASS
  lower_super_output_area                     39         39  PASS
  output_area                               1472       1472  PASS
  accident                                  8358       8358  PASS
  accident_road                            11714      11714  PASS

All row counts match.


In [42]:
def get_view_count(view_id):
    r = requests.get(
        f"{ENDPOINT}/api/v1/database/{DATABASE_ID}/view/{view_id}/data",
        params={"page": 0, "size": 1},
        auth=AUTH, headers=HEADERS, timeout=30
    )
    if not r.ok:
        return f"HTTP {r.status_code}"
    body = r.json()
    if isinstance(body, dict):
        return body.get("pagination", {}).get("total", body.get("total", "?"))
    elif isinstance(body, list):
        r2 = requests.get(
            f"{ENDPOINT}/api/v1/database/{DATABASE_ID}/view/{view_id}/data",
            params={"page": 0, "size": 100000},
            auth=AUTH, headers=HEADERS, timeout=60
        )
        body2 = r2.json()
        return len(body2) if isinstance(body2, list) else body2.get("pagination", {}).get("total", "?")
    return "?"

VIEW_EXPECTED = {
    "ml_accident_features":   8358,
    "ml_fatal_accidents":     int((df['Severity'] == 1).sum()),
    "ml_serious_accidents":   int((df['Severity'] == 2).sum()),
    "ml_rural_accidents":     int((df['Rural or Urban'] == 'Rural').sum()),
    "ml_high_speed_accidents":int((df['Speed_lim'] >= 60).sum()),
}

print(f"  {'View':<30s}  {'Local CSV':>10s}  {'DBRepo':>10s}  Match")
print("-" * 60)
view_pass = True
for vname, expected in VIEW_EXPECTED.items():
    vid = VIEW_IDS.get(vname)
    if not vid:
        print(f"  {vname:<30s}  {expected:>10d}  {'(no ID)':>10s}  SKIP")
        continue
    actual = get_view_count(vid)
    ok     = actual == expected
    if not ok:
        view_pass = False
    print(f"  {vname:<30s}  {expected:>10d}  {str(actual):>10s}  {'PASS' if ok else 'MISMATCH'}")

print()
print("All VIEWs verified." if view_pass else "Some VIEWs differ — check above.")


  View                             Local CSV      DBRepo  Match
------------------------------------------------------------
  ml_accident_features                  8358        8358  PASS
  ml_fatal_accidents                     197         197  PASS
  ml_serious_accidents                  1828        1828  PASS
  ml_rural_accidents                    5973        5973  PASS
  ml_high_speed_accidents               4970        4970  PASS

All VIEWs verified.
